# Phase-1 Only Training (Loc Augmentation Switchable/Repeat)
**What this notebook does**
- Per-class split: **70% train pool / 20% val / 10% test** (real-only val/test).
- Trains **only Phase 1** (natural imbalanced distribution; no balanced fine-tuning).
- Provides a **switchable Loc augmentation mode** via `LOC_AUG_MODE`.
- Lets you compare how different Loc-specific augmentation strengths affect performance.
- Saves the trained checkpoint and evaluation outputs.


In [ ]:

# ===== Imports & basic config =====
import os, random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import datasets as tvds, transforms as T, models
from torchvision.transforms import InterpolationMode

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt

import albumentations as A
import cv2

# ---- paths & high-level settings ----
DATA_ROOT = r"C:\Users\sangh\2025Fall\WM-811K_ImageFolder"   # <-- set your dataset root (ImageFolder layout)
RESULTS_DIR = Path("./phase1_only_loc_aug_results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- training & augmentation targets ----
IMG_SIZE = (64, 64)          # slightly larger helps small defects
# ---- split ratios (per class) ----
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.2, 0.1

# ---- training hyperparameters ----
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
USE_AMP = False              # keep off for tiny-S stability; turn on later if desired
FREEZE_EPOCHS = 2            # freeze backbone first few epochs to stabilize learning
BATCH_SIZE = 64
NUM_WORKERS = 0              # Windows-safe
SEED = 42

# ---- class options ----
DONT_AUGMENT_NAMES = {"unknown", "none"}  # set() to augment all classes

# ---- device & seeds ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)

# ===== Phase-1-only / Loc augmentation config =====
LOC_ALIASES = {"Loc", "local", "LOC"}
PHASE1_EPOCHS = EPOCHS
PHASE1_LR_MULT = 0.5

# Choose one of: "base", "mild", "moderate", "photo", "strong"
LOC_AUG_MODE = "mild"

# Optional probabilities / strengths can be tuned here if needed.


# ===== Build per-class split once (train_pool / val / test) =====
base = tvds.ImageFolder(DATA_ROOT)
CLASSES = base.classes
NUM_CLASSES = len(CLASSES)
print("CLASSES:", CLASSES)

dont_aug_ids = {i for i,c in enumerate(CLASSES) if c.lower() in DONT_AUGMENT_NAMES}

paths_per_class = defaultdict(list)
for p, y in base.samples:
    paths_per_class[y].append(p)

rng = np.random.RandomState(SEED + 111)
splits = {}
rows = []
for c_id, cname in enumerate(CLASSES):
    paths = paths_per_class[c_id]
    n = len(paths)
    if n < 3:
        raise RuntimeError(f"Class '{cname}' has only {n} images; need >=3 to split.")
    idx = np.arange(n); rng.shuffle(idx)
    n_train = int(round(TRAIN_RATIO * n))
    n_val   = int(round(VAL_RATIO   * n))
    n_test  = n - n_train - n_val
    if n_test < 0:
        n_test = max(0, n - n_train - n_val); n_val = max(0, n - n_train - n_test)
    train_idx = idx[:n_train]; val_idx = idx[n_train:n_train+n_val]; test_idx = idx[n_train+n_val:]
    splits[c_id] = {
        "train_pool": [paths[i] for i in train_idx],
        "val": [paths[i] for i in val_idx],
        "test":[paths[i] for i in test_idx],
    }
    rows.append({"class": cname, "train_pool": len(splits[c_id]["train_pool"]),
                 "val": len(splits[c_id]["val"]), "test": len(splits[c_id]["test"])})
print(pd.DataFrame(rows))

# ===== Dataset wrappers with class-conditional augmentation =====
from torch.utils.data import Dataset
from torchvision import transforms as T
from PIL import Image

def is_Loc_name(name):
    return name.lower().replace("-", " ").strip() in {s.replace("-", " ") for s in LOC_ALIASES}

def build_loc_aug(img_size=(64, 64), mode="mild"):
    if mode == "base":
        return T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation((0, 90), expand=False),
            T.ToTensor(),
        ])
    elif mode == "mild":
        return T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation((0, 90), expand=False),
            T.RandomApply([
                T.RandomAffine(
                    degrees=5,
                    translate=(0.02, 0.02),
                    scale=(0.98, 1.02),
                    shear=0
                )
            ], p=0.5),
            T.ToTensor(),
        ])
    elif mode == "moderate":
        return T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation((0, 90), expand=False),
            T.RandomAffine(
                degrees=8,
                translate=(0.04, 0.04),
                scale=(0.95, 1.05),
                shear=2
            ),
            T.ToTensor(),
        ])
    elif mode == "photo":
        return T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation((0, 90), expand=False),
            T.ColorJitter(brightness=0.05, contrast=0.05),
            T.RandomAffine(
                degrees=5,
                translate=(0.02, 0.02),
                scale=(0.98, 1.02),
                shear=0
            ),
            T.ToTensor(),
        ])
    elif mode == "strong":
        return T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation((0, 90), expand=False),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.RandomAffine(
                degrees=10,
                translate=(0.05, 0.05),
                scale=(0.95, 1.05),
                shear=5
            ),
            T.ToTensor(),
        ])
    else:
        raise ValueError(f"Unknown LOC_AUG_MODE={mode!r}. Use one of: "
                         f"'base', 'mild', 'moderate', 'photo', 'strong'.")

class TrainFromPaths(Dataset):
    def __init__(self, path_label_pairs, img_size=(64,64), class_names=None, loc_aug_mode="mild"):
        self.samples = path_label_pairs
        self.img_size = img_size
        self.class_names = class_names

        self.base_aug = T.Compose([
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation((0, 90), expand=False),
            T.ToTensor(),
        ])
        self.loc_aug = build_loc_aug(img_size=img_size, mode=loc_aug_mode)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        cname = self.class_names[label] if self.class_names is not None else ""
        if is_Loc_name(cname):
            x = self.loc_aug(img)
        else:
            x = self.base_aug(img)
        return x, label

class EvalFromPaths(Dataset):
    def __init__(self, path_label_pairs, img_size=(64,64)):
        self.samples = path_label_pairs
        self.tf = T.Compose([T.Resize(img_size), T.ToTensor()])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label

# ===== Builders for Phase-1-only Training =====
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np

def get_full_train_candidates():
    base = tvds.ImageFolder(DATA_ROOT)
    valset, testset = set(), set()
    for c_id in range(len(CLASSES)):
        for p in splits[c_id]["val"]:
            valset.add(p)
        for p in splits[c_id]["test"]:
            testset.add(p)
    paths_per_class = [[] for _ in CLASSES]
    for p, y in base.samples:
        if (p not in valset) and (p not in testset):
            paths_per_class[y].append(p)
    return paths_per_class

def build_phase1_loaders(batch_size=BATCH_SIZE, base_seed=SEED, loc_aug_mode=LOC_AUG_MODE):
    paths_per_class = get_full_train_candidates()
    train_pairs = []
    val_parts, test_parts = [], []

    for c_id in range(len(CLASSES)):
        train_pairs.extend([(p, c_id) for p in paths_per_class[c_id]])
        val_parts.append(EvalFromPaths([(p, c_id) for p in splits[c_id]["val"]], img_size=IMG_SIZE))
        test_parts.append(EvalFromPaths([(p, c_id) for p in splits[c_id]["test"]], img_size=IMG_SIZE))

    train_ds = TrainFromPaths(
        train_pairs,
        img_size=IMG_SIZE,
        class_names=CLASSES,
        loc_aug_mode=loc_aug_mode
    )
    val_ds = ConcatDataset(val_parts)
    test_ds = ConcatDataset(test_parts)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(base_seed)
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    return train_loader, val_loader, test_loader

def seed_worker(worker_id):
    base_seed = torch.initial_seed() % 2**32
    np.random.seed(base_seed + worker_id)
    random.seed(base_seed + worker_id)


# ===== Model, evaluation, and training loop (stable) =====
def make_resnet18(num_classes, pretrained=True, dropout=0.2):
    if pretrained:
        try:
            weights = models.ResNet18_Weights.IMAGENET1K_V1
        except AttributeError:
            weights = "IMAGENET1K_V1"
        net = models.resnet18(weights=weights)
    else:
        net = models.resnet18(weights=None)
    in_feats = net.fc.in_features
    net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feats, num_classes)) if dropout else nn.Linear(in_feats, num_classes)
    return net.to(DEVICE)

@torch.no_grad()
def evaluate_model(model, loader, tag, save=True):
    model.eval()
    ys, yh = [], []
    for x,y in loader:
        x = x.to(DEVICE); out = model(x)
        ys.extend(y.tolist()); yh.extend(out.argmax(1).cpu().tolist())
    ys = np.array(ys); yh = np.array(yh)
    cm = confusion_matrix(ys, yh, labels=list(range(NUM_CLASSES)))
    per_class_acc = np.diag(cm) / np.clip(cm.sum(1), 1, None)
    prec, rec, f1, sup = precision_recall_fscore_support(ys, yh, labels=list(range(NUM_CLASSES)), zero_division=0)
    overall = float((ys==yh).mean())

    if save:
        (RESULTS_DIR / "runs").mkdir(exist_ok=True)
        pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(RESULTS_DIR / f"{tag}_cm.csv")
        pd.DataFrame({"class": CLASSES, "acc": per_class_acc, "precision": prec, "recall": rec, "f1": f1, "support": sup}).to_csv(RESULTS_DIR / f"{tag}_per_class.csv", index=False)
        # PNG
        plt.figure(figsize=(8,6)); plt.imshow(cm, interpolation='nearest', aspect='auto')
        plt.title(f"Confusion Matrix ({tag})"); plt.colorbar()
        ticks = np.arange(len(CLASSES)); plt.xticks(ticks, CLASSES, rotation=45, ha='right'); plt.yticks(ticks, CLASSES)
        thresh = cm.max()/2 if cm.size else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                v = int(cm[i,j]); plt.text(j, i, str(v), ha='center', va='center', color='white' if v>thresh else 'black', fontsize=8)
        plt.ylabel('True'); plt.xlabel('Pred'); plt.tight_layout(); plt.savefig(RESULTS_DIR / f"{tag}_cm.png", dpi=150); plt.close()
    return {"overall_acc": overall, "per_class_acc": per_class_acc, "precision": prec, "recall": rec, "f1": f1, "support": sup}

def fit_and_report(model, train_loader, val_loader, test_loader, tag, weights=None,
                   epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY, label_smooth=LABEL_SMOOTH,
                   use_amp=USE_AMP, freeze_epochs=FREEZE_EPOCHS):
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smooth)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler() if (use_amp and torch.cuda.is_available()) else None

    # freeze backbone for first few epochs
    def set_backbone_requires_grad(flag: bool):
        for n,p in model.named_parameters():
            if n.startswith("fc."):
                p.requires_grad = True
            else:
                p.requires_grad = flag
    if freeze_epochs > 0:
        set_backbone_requires_grad(False)

    best_val, best_state, bad, patience = -1.0, None, 0, 3
    history_rows = []

    for ep in range(1, epochs+1):
        if freeze_epochs > 0 and ep == (freeze_epochs+1):
            set_backbone_requires_grad(True)

        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x,y in train_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    out = model(x); loss = criterion(out, y)
                if not torch.isfinite(loss):
                    print("Non-finite loss; skipping batch."); continue
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                out = model(x); loss = criterion(out, y)
                if not torch.isfinite(loss):
                    print("Non-finite loss; skipping batch."); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            loss_sum += float(loss.item())
            pred = out.argmax(1); total += y.size(0); correct += (pred==y).sum().item()

        train_loss = loss_sum / max(1, len(train_loader)); train_acc = correct / max(1, total)
        val_metrics = evaluate_model(model, val_loader, tag=f"{tag}_val_tmp", save=False)
        history_rows.append({"epoch": ep, "train_loss": train_loss, "train_acc": train_acc, "val_acc": val_metrics["overall_acc"]})
        print(f"Epoch {ep:02d}/{epochs}  train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  val_acc={val_metrics['overall_acc']:.3f}")

        scheduler.step()
        curr = val_metrics["overall_acc"]
        if curr > best_val + 1e-4:
            best_val, bad = curr, 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            bad += 1
            # Early stopping disabled: do NOT break
            # if bad >= patience:
            #     print(f"Early stopping at epoch {ep} (best val={best_val:.3f})")
            #     break


    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k,v in best_state.items()})

    val_final = evaluate_model(model, val_loader, tag=f"{tag}_VAL", save=True)
    test_final= evaluate_model(model, test_loader, tag=f"{tag}_TEST", save=True)

    pd.DataFrame(history_rows).to_csv(RESULTS_DIR / f"{tag}_history.csv", index=False)
    pd.DataFrame([{"val_acc": val_final["overall_acc"], "test_acc": test_final["overall_acc"]}]).to_csv(RESULTS_DIR / f"{tag}_summary.csv", index=False)
    return {"val": val_final, "test": test_final}

# ===== Phase-1-only training =====
print(f"Running Phase-1-only training with LOC_AUG_MODE = {LOC_AUG_MODE!r}")

train1, val1, test1 = build_phase1_loaders(
    base_seed=SEED + 101,
    loc_aug_mode=LOC_AUG_MODE
)

model = make_resnet18(NUM_CLASSES, pretrained=True, dropout=0.2)

print("\n=== Phase 1 only: Imbalanced training ===")
lr1 = LR * PHASE1_LR_MULT
out1 = fit_and_report(
    model, train1, val1, test1, tag=f"P1_ONLY_{LOC_AUG_MODE.upper()}",
    epochs=PHASE1_EPOCHS, lr=lr1, wd=WEIGHT_DECAY,
    label_smooth=LABEL_SMOOTH, use_amp=USE_AMP, freeze_epochs=FREEZE_EPOCHS
)

(RESULTS_DIR / "checkpoints").mkdir(exist_ok=True)
ckpt_p1 = RESULTS_DIR / "checkpoints" / f"P1_ONLY_{LOC_AUG_MODE.upper()}_resnet18.pt"
torch.save(
    {"model": model.state_dict(), "classes": CLASSES, "img_size": IMG_SIZE,
     "tag": f"P1_ONLY_{LOC_AUG_MODE.upper()}", "loc_aug_mode": LOC_AUG_MODE},
    ckpt_p1
)
print(f"Saved Phase-1-only checkpoint to: {ckpt_p1}")
